<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **8. Hardware for Local LLMs**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 8.1 Introduction

[05.07_Introduction_to_Local_LLMs.ipynb](05.07_Introduction_to_Local_LLMs.ipynb) looked at the models on your Pi. This notebook looks at the chips that run them. Almost all the work of a language model is one kind of arithmetic, the multiplication of large matrices, and three kinds of processors do it in very different ways: the **CPU**, the main processor of every computer, the **GPU**, built for graphics and now used for AI, and the **NPU**, built for neural networks and nothing else.

You will watch all three multiply the same matrices, measure your own laptop, look inside the Pi over SSH, and find out what actually limits the speed of a language model on it. The last part compares the Pi with a laptop, a phone and a data centre. Allow about **70 minutes**, plus time for the questions.

By the end, you should be able to explain how a CPU, a GPU and an NPU organise the same multiplications, measure how fast a computer calculates and how fast it reads its memory, tell whether a task is limited by arithmetic or by memory, and estimate from a device's specifications how fast a model could run on it.

#### 8.1.1 Environment Setup

Sections 8.2 and 8.3 run on your laptop and use `numpy` and `torch`. Section 8.4 needs the Pi, connected as in [05.04_GettingStarted_with_Raspberry_Pi.ipynb](05.04_GettingStarted_with_Raspberry_Pi.ipynb), and a terminal for SSH (05.04, Section 4.7).

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

In [ ]:
import html
import os
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

FIGURES = Path("../../.admin/notebooks/hardware")

### 8.2 One Operation, Three Designs

A language model spends nearly all of its time multiplying matrices (05.03, 05.05). Each entry of the result is built from one small step, repeated many times: the **multiply-accumulate**, or MAC. It multiplies two numbers and adds the product to a running sum:

$$
s \leftarrow s + a \times b
$$

With $s = 10$, $a = 3$ and $b = 4$, one MAC turns the sum into $10 + 3 \times 4 = 22$. One entry of the product $C = AB$ of two $n \times n$ matrices needs $n$ MACs, one for each pair of numbers from a row of $A$ and a column of $B$, so the whole product needs $n^3$. Each MAC counts as two operations, a multiplication and an addition, and a matrix product costs $2n^3$ operations. That is the unit behind every speed in this notebook: FLOP/s for floating-point operations per second, TOPS for trillions of operations per second.

All three processors compute exactly this. They differ in how many MACs they do at once, and in how often they have to fetch the numbers from memory [07].

<center>
<img src="../../.admin/notebooks/hardware/processor-architecture.png"
     alt="How a CPU, a GPU and an NPU organise computation and memory"
     width="90%">
</center>

- A **CPU** has a few strong cores. Each one runs any program, with decisions, branches and calls to the operating system, and each has a vector unit that does a handful of MACs at once.
- A **GPU** has thousands of simple units that all run the same small program on different data. It is fast when the work splits into many equal pieces, as a matrix product does.
- An **NPU** is a grid of MAC units. A compiler turns a neural network into a fixed schedule for that grid, and whatever the compiler does not support, the NPU cannot run.

The animations below multiply two 16 × 16 matrices, one design at a time. Press Play in each. Their speeds are chosen for watching, not measured.

#### 8.2.1 CPU: one entry at a time

The CPU view works through the 256 entries of C one after the other, each with 16 MACs. Real CPU libraries do better: the vector unit works on several entries at once, and the cores share the matrix between them. Section 8.3 shows how much that is worth. This view would take about five minutes to finish, so watch it for twenty seconds, then move on.

In [ ]:
def animation(name, height=500):
    """One of the matrix animations, played inside the notebook."""
    page = html.escape((FIGURES / f"{name}.html").read_text(encoding="utf-8"))
    frame = (f'<iframe sandbox="allow-scripts" srcdoc="{page}" '
             f'style="width:100%;height:{height}px;border:1px solid #cbd5de;border-radius:8px"></iframe>')
    display({"text/html": frame}, raw=True)


animation("cpu-matrix-16x16")

#### 8.2.2 GPU: many entries at once

The GPU view splits C into tiles of 4 × 4 entries, and the 16 entries of a tile grow together. A real GPU works on thousands of such pieces at the same time.

In [ ]:
animation("gpu-matrix-16x16")

#### 8.2.3 NPU: numbers flowing through a grid

The NPU view is a grid of 4 × 4 MAC units, a design called a **systolic array** [01]. Each unit keeps one entry of C. Numbers from A move to the right and numbers from B move down, one step per tick, and each unit multiplies the two numbers passing through it and adds the product to its sum. That is the diagonal wave you see in C.

All three views use each number many times, but they get it to the units in different ways. The CPU view fetches a number of A again for every entry of C that needs it, 16 times in all. The GPU and the NPU views fetch it once per tile and use it for four entries. In the GPU, the units of a group read it from a small memory they share. In the NPU, each unit hands it to its neighbour, so the units never have to look anything up. Moving a number costs far more time and energy than multiplying it, and handing it next door is the cheapest way to move it. Google's TPUs, the chips behind its own AI services, are built on the same idea [02].

In [ ]:
animation("npu-matrix-16x16", height=530)

#### 8.2.4 Who does what

| Processor | Built from | Strong at | Weak at | Found in |
|:----------|:-----------|:----------|:--------|:---------|
| **CPU** | a few strong cores, each with a vector unit | any program, decisions, small jobs | large matrix products | every computer and phone |
| **GPU** | thousands of simple units | large matrix products, training, serving many users | small jobs, power use | data centres, gaming PCs, and built into most laptops and phones |
| **NPU** | a grid of MAC units, driven by a compiler | neural networks at low power | anything its compiler does not support, training | phones, newer laptops, add-on boards for the Pi |
| **TPU** | Google's large systolic arrays | training and serving very large models | anything that is not a large neural network | Google's data centres, rented through Google Cloud |

Processors also differ in their instruction set, the basic commands they understand. Most PCs and servers use **x86**, from Intel and AMD. Phones, the Pi, Apple's Macs and more and more Windows laptops use **ARM**, and most ARM chips are designed for low power. In phones, in the Pi and in most laptops, the CPU and the GPU built into the main chip share one memory, while Apple's M chips add the NPU to it.

### 8.3 Your Laptop

First, what is in the computer in front of you. `platform` tells you the instruction set, and `os.cpu_count()` how many cores the operating system sees. That can be twice the number of physical cores, and many laptops mix fast cores with slower, power-saving ones. `torch` tells you whether there is a GPU it can use. Plug your laptop in for this section: on battery, many laptops slow down.

In [ ]:
print("instruction set: ", platform.machine())
print("operating system:", platform.platform())
print("cores the system sees:", os.cpu_count())

if torch.cuda.is_available():
    GPU, sync = "cuda", torch.cuda.synchronize
    print("GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    GPU, sync = "mps", torch.mps.synchronize
    print("GPU: the Apple GPU, through Metal")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    GPU, sync = "xpu", torch.xpu.synchronize
    print("GPU:", torch.xpu.get_device_name(0))
else:
    GPU = None
    print("GPU: none that torch can use")

`x86_64` or `AMD64` means x86, `arm64` or `aarch64` means ARM. Most laptops have a GPU built into the main chip, and newer ones an NPU as well, but torch can only use a GPU if it was installed with the software for it: CUDA for NVIDIA (AMD's ROCm also shows up as `cuda`), Metal on a Mac, XPU for Intel. If it finds none, that is normal, and Section 8.3.2 simply skips. To see what your laptop really has, look at Task Manager, Performance on Windows (it lists CPU, GPU and NPU), About This Mac on a Mac, or `lscpu` and `lspci` on Linux.

#### 8.3.1 Calculating

We start with the slowest way to multiply matrices: plain Python, one MAC after the other, as in the CPU animation. Then the same with `numpy`, which hands the work to a library written for this one job.

<div class="alert alert-block alert-info">

Predict before you run: how many times faster will numpy be?
</div>

In [ ]:
def best_time(work, repeat=3):
    """The fastest of a few runs, in seconds."""
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        work()
        times.append(time.perf_counter() - start)
    return min(times)


def matmul_python(A, B):
    n = len(A)
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n)] for i in range(n)]


N = 100
A, B = np.random.rand(N, N), np.random.rand(N, N)
slow = best_time(lambda: matmul_python(A.tolist(), B.tolist()), repeat=1)
fast = best_time(lambda: A @ B, repeat=10)
print(f"plain Python: {slow * 1000:9.3f} ms, {2 * N**3 / slow / 1e9:7.2f} GFLOP/s")
print(f"numpy:        {fast * 1000:9.3f} ms, {2 * N**3 / fast / 1e9:7.2f} GFLOP/s")
print(f"numpy is {slow / fast:.0f} times faster")

The arithmetic did not change. numpy's library is compiled instead of interpreted, keeps the vector unit of every core busy, works on blocks of the matrix that fit into the caches, and shares the work between all cores. The hardware was always there, it is just that the software decides how much of it is used.

Now the speed your laptop reaches on larger matrices, with 32-bit floating-point numbers (FP32):

In [ ]:
laptop_gflops = 0
for n in (256, 1024, 3000):
    A = np.random.rand(n, n).astype(np.float32)
    B = np.random.rand(n, n).astype(np.float32)
    gflops = 2 * n**3 / best_time(lambda: A @ B) / 1e9
    laptop_gflops = max(laptop_gflops, gflops)
    print(f"{n:5d} x {n:<5d} {gflops:7.0f} GFLOP/s")

Small matrices usually reach less: starting the work costs a fixed amount of time, and there is too little of it to keep every core busy.

#### 8.3.2 The GPU, if you have one

If torch found a GPU, the next cell multiplies the same matrices there, once with the numbers already in the GPU's memory and once including the copy there and back. Without a GPU, it says so and does nothing else.

<div class="alert alert-block alert-info">

Predict: from which size on is the GPU faster than numpy on the CPU? Does the copy change the answer? Try other sizes in `SIZES`.
</div>

In [ ]:
SIZES = (64, 512, 2048, 4096)

laptop_gpu_gflops = None
if GPU is None:
    print("No GPU that torch can use. Continue with Section 8.3.3.")
else:
    for n in SIZES:
        A = np.random.rand(n, n).astype(np.float32)
        B = np.random.rand(n, n).astype(np.float32)
        A_gpu, B_gpu = torch.from_numpy(A).to(GPU), torch.from_numpy(B).to(GPU)

        def on_gpu():
            A_gpu @ B_gpu
            sync()

        def with_copy():
            (torch.from_numpy(A).to(GPU) @ torch.from_numpy(B).to(GPU)).cpu()

        on_gpu()   # the first call sets the GPU up; it does not count
        cpu, gpu, copy = (2 * n**3 / best_time(work) / 1e9 for work in (lambda: A @ B, on_gpu, with_copy))
        laptop_gpu_gflops = max(laptop_gpu_gflops or 0, gpu)
        print(f"{n:5d} x {n:<5d} CPU {cpu:8.0f}   GPU {gpu:8.0f}   GPU with copy {copy:8.0f} GFLOP/s")

A GPU wins on large matrices, often by far. On small ones it loses: every piece of work has to be prepared and handed to the GPU, and the numbers have to travel there and back. That is why a language model on a GPU keeps its weights in the GPU's memory. Where CPU and GPU share one memory, software written for it can skip the copy, though torch still makes one, also on a Mac.

#### 8.3.3 Reading memory

Calculating is one speed limit. The other is how fast the processor reads its memory. The next cell adds up 400 MB of numbers. Adding is cheap, so the time goes almost entirely into reading, and the result is the memory bandwidth in GB per second. torch uses all cores for it.

In [ ]:
numbers = torch.ones(100_000_000)   # 100 million 32-bit numbers, 400 MB
laptop_gbs = numbers.numel() * 4 / best_time(lambda: numbers.sum()) / 1e9
print(f"your laptop reads about {laptop_gbs:.0f} GB per second")
del numbers

Keep both numbers, `laptop_gflops` and `laptop_gbs`. Section 8.5 puts them next to the Pi, a phone and a data centre.

### 8.4 Your Pi

#### 8.4.1 Look inside

Open a terminal on your laptop and log in to the Pi as in 05.04, Section 4.7:

```sh
ssh cas@172.31.250.1
```

Then, once you see the prompt of the Pi, run these commands one after the other:

```sh
pinout
lscpu
free -h
vcgencmd measure_clock arm
vcgencmd measure_temp
```

- `pinout` draws the board and lists what is on it: the main chip (SoC, a system on a chip: CPU, GPU and more in one), the memory, the RP1 chip that runs USB, Ethernet and the row of pins along the edge, and what each pin is for.
- `lscpu` describes the processor: the instruction set (`aarch64`, 64-bit ARM), four Cortex-A76 cores, their clock between 1500 and 2400 MHz, and the caches, small and fast memories inside the chip. The line `Flags` lists what the cores can do beyond plain arithmetic. `asimd` is the vector unit (Arm calls it NEON), `asimdhp` adds 16-bit floating-point numbers, and `asimddp` adds dot products of 8-bit integers, which quantized models use.
- `free -h` shows the 8 GB of memory and how much of it the models take right now.
- `vcgencmd` asks the chip's firmware: `measure_clock arm` gives the clock of the cores in Hz, `measure_temp` the temperature. The Pi lowers its clock when it has nothing to do and raises it under load.

<center>
<img src="../../.admin/notebooks/hardware/pi5-inside.png"
     alt="Inside the Raspberry Pi 5 chip: four cores with their caches, the GPU, and the shared memory"
     width="85%">
</center>

The caches are tiny next to a model. All of them together hold about 4.5 MB; `granite4.2:3b` is a 2.2 GB file.

#### 8.4.2 Two speed limits

`lscpu` gives the cores and the clock; the rest comes from the design of the chip and its memory [03]. Calculating: each core can start two vector instructions per clock tick, and each of them multiplies and adds four 32-bit numbers, so 16 operations per tick. With 8-bit integers, the dot-product instructions do up to four times as many. Reading: the memory transfers 4 bytes 4267 million times per second.

In [ ]:
CORES, CLOCK = 4, 2.4e9                     # from lscpu
PI_GFLOPS = CORES * CLOCK * 16 / 1e9        # 32-bit floating point
PI_GOPS_INT8 = CORES * CLOCK * 64 / 1e9     # 8-bit integers, with dot products
PI_GBS = 4267e6 * 4 / 1e9                   # LPDDR4X-4267, 4 bytes wide

print(f"calculating: {PI_GFLOPS:.0f} GFLOP/s with 32-bit floats, {PI_GOPS_INT8:.0f} GOP/s with 8-bit integers")
print(f"reading:     {PI_GBS:.1f} GB/s")

#### 8.4.3 Reading, writing and watts

A model does two kinds of work. In the **prompt** phase it processes all tokens of your prompt together: each weight is fetched once and used for hundreds of tokens. In the **answer** phase it writes one token at a time, and a dense model like granite needs every weight once for every token. Two limits, two phases: which limit holds back which? This view of performance is called the roofline model [06].

The board also measures its own power. In the terminal on the Pi, start this loop. Every second, it prints the power of the chip and its memory in watts, until you press Ctrl+C (also on a Mac; in Windows Terminal, click into the window first, or Ctrl+C copies the selected text instead):

```sh
while true; do vcgencmd pmic_read_adc | awk '/current/ {split($2,a,"="); sub("A","",a[2]); i[substr($1,1,length($1)-2)]=a[2]} /volt/ {split($2,v,"="); sub("V","",v[2]); u[substr($1,1,length($1)-2)]=v[2]} END {for (r in i) w+=i[r]*u[r]; printf "%.1f W\n", w}'; sleep 1; done
```

The power chip of the board measures the current and the voltage of each supply line, and `awk` multiplies and adds them. USB devices, the fan and the losses of the power supply are not included.

<div class="alert alert-block alert-info">

Put the terminal next to this notebook and note the watts while the Pi does nothing. Predict which limit holds back each phase. Then run the next cell, which gives `granite4.2:3b` the first 3000 characters of `caspi.py` to summarise. Watch the watts during the prompt phase, while nothing appears yet, and during the answer phase, while the answer appears word by word. It takes about a minute, plus up to two minutes if the model has to be loaded first.
</div>

In [ ]:
import caspi

caspi.connect()
MODEL = "granite4.2:3b"
caspi.load(MODEL)
info = next(m for m in caspi.models(info=True) if m["name"] == MODEL)
PARAMETERS, FILE_GB = float(info["parameters"][:-1]) * 1e9, info["size_gb"]

# a new first line on every run, so that Ollama cannot reuse the prompt from its cache
prompt = (f"Run {time.time():.0f}.\n" + open("caspi.py", encoding="utf-8").read()[:3000]
          + "\n\nSummarise this code in three sentences.")
stats = caspi.ask(MODEL, prompt, think=False, temperature=0, num_predict=150).stats

print()
for phase, speed in [("prompt", stats["prompt_tokens_per_s"]), ("answer", stats["reply_tokens_per_s"])]:
    calculated = 2 * PARAMETERS * speed / 1e9   # two operations per weight and token
    print(f"{phase}: {speed:5.1f} tokens/s, {calculated:4.0f} GOP/s = {calculated / PI_GOPS_INT8:4.0%} of the integer limit, "
          f"{calculated / PI_GFLOPS:4.0%} of the float limit")
read = FILE_GB * stats["reply_tokens_per_s"]   # a dense model reads the whole file for every token
print(f"answer: {read:.1f} GB/s read from memory = {read / PI_GBS:.0%} of the memory limit")
print(f"answer: {2 * PARAMETERS / (FILE_GB * 1e9):.1f} operations per byte read; "
      f"the cores could do {PI_GOPS_INT8 / PI_GBS:.0f} in the time memory delivers one byte")

The prompt phase keeps the cores busy. It calculates several times as much per second as the answer phase, often even more than the cores could with 32-bit floats, because the quantized weights are multiplied as 8-bit integers. Memory hardly matters there, since each weight is fetched once for hundreds of tokens. The answer phase calculates only a few percent of what the cores can, but uses well over half of what the memory can deliver. The last line says why: it does about 3 operations for every byte it reads, while the cores could do about 36 in the time it takes the memory to deliver one byte. Below that ratio, memory sets the speed. `htop` would show the cores busy in both phases, because a core that waits for memory still counts as busy. That is why the speed of the answer follows the size of the model file ([05.09_Making_Models_Smaller.ipynb](05.09_Making_Models_Smaller.ipynb) checks this on three models), and why faster cores alone would not make the Pi write faster.

The watts tell the same story. In our runs the chip drew about 2 W at rest, about 8.5 W in the prompt phase, with all cores calculating, and about 6.5 W in the answer phase, with the cores waiting much of the time.

<div class="alert alert-block alert-info">

Put the watts you saw during the answer phase into the next cell. What does one written token cost in energy?
</div>

In [ ]:
WATTS_ANSWER = 6.5   # your reading during the answer phase

joules = WATTS_ANSWER / stats["reply_tokens_per_s"]
print(f"{joules:.2f} joules per written token")
print(f"a phone battery of 15 Wh would last for about {15 * 3600 / joules:,.0f} tokens")

### 8.5 Pi, Laptop, Phone and Data Centre

#### 8.5.1 GPU and NPU on a Pi

Your Pi has a GPU, but Ollama does not use it, and it has no NPU. We measured both on a Pi 5 with an AI HAT+ 2, an add-on board with a Hailo-10H NPU and 8 GB of memory of its own [04]. There were three tasks: a 512 × 512 matrix product with 32-bit floats, written the same way for CPU and GPU. ResNet-50, an image classifier with 26 million parameters, labelling one photo, and Qwen2-1.5B, a small language model, answering short questions. The NPU ran quantized versions of the two models. All four ways gave the photo the same label, a Samoyed.

In [ ]:
LAB = {   # Raspberry Pi 5 with AI HAT+ 2
    "matrix product, ms (shorter is better)": {"CPU, 1 core": 39.5, "CPU, 4 cores": 10.1, "GPU": 96.4},
    "ResNet-50, ms per photo (shorter is better)": {"CPU, 1 core": 217.8, "CPU, 4 cores": 74.8, "GPU": 393.1, "NPU": 4.5},
    "Qwen2-1.5B, tokens/s (longer is better)": {"CPU, 4 cores": 8.5, "GPU": 0.27, "NPU": 7.1},
}
COLOURS = {"CPU": "#2463a6", "GPU": "#a55016", "NPU": "#087f72"}

fig, axes = plt.subplots(1, 3, figsize=(15, 3), constrained_layout=True)
for ax, (task, results) in zip(axes, LAB.items()):
    ax.barh(list(results), list(results.values()), color=[COLOURS[name[:3]] for name in results])
    for y, value in enumerate(results.values()):
        ax.text(value, y, f" {value:g}", va="center")
    ax.set_xscale("log")
    ax.set_title(task, loc="left", fontsize=10)
    ax.invert_yaxis()
plt.show()

The NPU labels the photo about 17 times faster than all four CPU cores, while drawing a few watts. ResNet-50 is small and does a lot of arithmetic for every weight it fetches, exactly the work a grid of MAC units was built for. For the language model, the NPU is even slower than the four CPU cores, 7.1 against 8.5 tokens per second. Writing text needs every weight for every token, so more MAC units do not help, and the NPU works within about 3 watts, while the Pi's processor may draw up to 10 [11]. What the NPU offers for text is that it leaves the CPU free for other work. The Pi's GPU loses every time. It was designed for graphics and video, and the software that runs general calculations on it is still young.

#### 8.5.2 From a phone to a data centre

The same two limits decide the speed of every device. The next table lists a few of them [08, 09, 10], with your laptop added from Section 8.3. Your laptop's numbers are measured; the others are the makers' peak values, which real software rarely reaches (your Pi reached about 12 of its 17 GB/s). The compute numbers come in different formats, and a number without its format means little: the AI HAT+ 2 reaches 40 TOPS with 4-bit integers, but 20 TOPS with 8-bit ones [04]. Watts and prices are rough, and the watts of the Pi are those of the chip in Section 8.4.3.

In [ ]:
LAPTOP_MEMORY_GB = 16   # Task Manager on Windows, About This Mac, or free -g on Linux

rows = [
    ("Raspberry Pi 5, CPU", PI_GFLOPS / 1000, "FP32", 8, PI_GBS, 8, 100),
    ("Raspberry Pi 5 with AI HAT+ 2, NPU", 40, "INT4", 8, None, 2.5, 230),
    ("iPhone 16 Pro, NPU", 35, "not stated", 8, None, None, 999),
    ("your laptop, CPU", laptop_gflops / 1000, "FP32", LAPTOP_MEMORY_GB, laptop_gbs, None, None),
    ("Copilot+ laptop, NPU (minimum)", 40, "not stated", 16, None, None, None),
    ("MacBook Air M4, NPU", 38, "not stated", 16, 120, None, 999),
    ("NVIDIA H100, data-centre GPU", 989, "BF16", 80, 3350, 700, 30000),
]
if laptop_gpu_gflops:
    rows.insert(4, ("your laptop, GPU", laptop_gpu_gflops / 1000, "FP32", None, None, None, None))
DEVICES = pd.DataFrame(rows, columns=["device", "TOPS", "format", "memory GB", "GB/s", "watts", "USD"]).set_index("device")

MODEL_GB = FILE_GB   # granite4.2:3b from Section 8.4.3: a dense model reads its whole file for every token
DEVICES["tokens/s at most"] = (DEVICES["GB/s"] / MODEL_GB).round(1)
DEVICES["joules per token, at least"] = (DEVICES["watts"] / DEVICES["tokens/s at most"]).round(2)
DEVICES["holds a 70B model"] = DEVICES["memory GB"] >= 70 * 4.8 / 8   # 4-bit weights: about 42 GB
DEVICES.round(2)

The upper limit for the answer follows from the bandwidth alone: bytes per second divided by the bytes read per token [06]. The data-centre GPU reads 200 times faster than the Pi and holds models that fit nowhere else, and even for a single user its energy per token can be lower than the Pi's. It also serves many users at once. When it writes the next token for 100 conversations together, it reads each weight once and uses it 100 times, so it calculates more per byte, as the Pi does in the prompt phase. That is why serving a model to many people is cheapest in a data centre. The Pi has other strengths: your data stays on it, there is no cost per token, and it needs no network (05.07).

<div class="alert alert-block alert-info">

**Optional: your phone.** Phones have NPUs too. Install the free app Geekbench AI [05] on your phone and run it once on the CPU and once on the NPU (on an iPhone it is called the Neural Engine). It runs the same small neural networks on both and gives a score for each. If you like, run it on your laptop as well, if it has an NPU that Geekbench AI supports. How much faster is the NPU, and does the phone or the laptop win?
</div>

### 8.6 Summarizing Remarks

A CPU, a GPU and an NPU all compute the same multiply-accumulates. The CPU runs anything, with a few strong cores; the GPU runs one small program on thousands of units; the NPU fixes its schedule in advance and reuses every number it fetches. Which one is fastest depends on the job and on the software: numpy was about a thousand times faster than plain Python on the same processor, and the Pi's GPU lost to its CPU everywhere.

For language models, the second limit matters more than the first. The prompt phase is arithmetic, which means that for a dense model the answer phase is reading the whole model from memory. That is why smaller and quantized models are faster ([05.09](05.09_Making_Models_Smaller.ipynb)), why the NPU on the Pi writes even more slowly than its CPU, and why a data centre, which reads each weight once for many users, is hard to beat on cost. A local model wins on other grounds: privacy, no network, and no cost per token.

### 8.7 Further Reading and Literature

[01] Kung, H. T. (1982). **Why Systolic Architectures?** IEEE Computer, 15(1), 37-46. [Paper](https://doi.org/10.1109/MC.1982.1653825)

[02] Jouppi, N. P., Young, C., Patil, N., et al. (2017). **In-Datacenter Performance Analysis of a Tensor Processing Unit.** ISCA. [Paper](https://arxiv.org/abs/1704.04760)

[03] Raspberry Pi. **Processors: BCM2712.** Raspberry Pi documentation. [Website](https://www.raspberrypi.com/documentation/computers/processors.html)

[04] Raspberry Pi. **AI HATs.** Raspberry Pi documentation. [Website](https://www.raspberrypi.com/documentation/accessories/ai-hat-plus.html)

[05] Primate Labs. **Geekbench AI.** [Website](https://www.geekbench.com/ai/)

[06] Williams, S., Waterman, A., & Patterson, D. (2009). **Roofline: An Insightful Visual Performance Model for Multicore Architectures.** Communications of the ACM, 52(4). [Paper](https://doi.org/10.1145/1498765.1498785)

[07] Sze, V., Chen, Y.-H., Yang, T.-J., & Emer, J. (2017). **Efficient Processing of Deep Neural Networks: A Tutorial and Survey.** Proceedings of the IEEE. [Paper](https://arxiv.org/abs/1703.09039)

[08] NVIDIA. **H100 Tensor Core GPU.** [Website](https://www.nvidia.com/en-us/data-center/h100/)

[09] Apple (2024). **Apple introduces M4 chip.** Newsroom. [Website](https://www.apple.com/newsroom/2024/05/apple-introduces-m4-chip/)

[10] Apple (2024). **Apple debuts iPhone 16 Pro and iPhone 16 Pro Max.** Newsroom. [Website](https://www.apple.com/newsroom/2024/09/apple-debuts-iphone-16-pro-and-iphone-16-pro-max/)

[11] Geerling, J. (2026). **Raspberry Pi's new AI HAT adds 8GB of RAM for local LLMs.** Blog post, with CPU and NPU measured on the same Pi 5. [Website](https://jeffgeerling.com/blog/2026/raspberry-pi-ai-hat-2)

---
### 8.8 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: Counting operations**

How many MACs and how many operations does a 512 × 512 matrix product take? One core of the lab Pi needed 39.5 ms for it. How many GFLOP/s is that, and how far is it from the limit of one core, a quarter of the 154 GFLOP/s in Section 8.4.2?

(answer here)

**Question 02: Three designs**

In the three animations, how many times is each number of A fetched? How does a number reach the units that need it in the GPU, and how in the NPU? Why does the NPU's way save energy?

(answer here)

**Question 03: Python and numpy**

Report your times and the factor from Section 8.3.1, and the best GFLOP/s of your laptop. Name two reasons why numpy is so much faster on the same processor.

(answer here)

**Question 04: Your laptop's GPU**

If torch found a GPU: from which size on did it beat the CPU, and what did the copy change? If not: which processor does your laptop have, and does it contain a GPU or an NPU that torch cannot use?

(answer here)

**Question 05: Inside the Pi**

Report the cores, the clock range and the caches from `lscpu`. How many times larger than all the caches together is the file of `granite4.2:3b`? What follows for the answer phase?

(answer here)

**Question 06: Two limits**

Report the speed of the prompt and of the answer phase in Section 8.4.3, in tokens per second and as a share of the Pi's limits. Which limit holds back which phase, and which line of the output shows it best?

(answer here)

**Question 07: Watts and joules**

Report the watts at rest, in the prompt phase and in the answer phase, and the joules per written token. Estimate the joules per token for a dense model with half the file size, and explain your estimate.

(answer here)

**Question 08: The lab Pi**

Why is the NPU about 17 times faster for the photo but slower than the CPU for text? Why did the Pi's GPU lose every time?

(answer here)

**Question 09: Choosing hardware**

Use the table of Section 8.5.2 to choose hardware for three cases: a private assistant for one family, a chatbot for 1000 employees, and a battery-powered camera that recognises birds. Justify each choice with at least one number.

(answer here)

**Question 10: Optional: your phone**

Report the Geekbench AI scores of your phone for CPU and NPU, and of your laptop if you ran it. How much did the NPU gain, and what does that tell you about the work it was built for?

(answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)